In [14]:
import os
import subprocess
from pathlib import Path
import onnxruntime as ort
from transformers import AutoTokenizer
import numpy as np

In [19]:
# 1.2. Define Paths
# IMPORTANT: These must match your system paths exactly.
CHECKPOINT_PATH = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/TrainSil/mobilebert-can-attack-classifier-experimental"
ONNX_OUTPUT_DIR = "OnnxModels/TrainSilOnnx" # Relative path will create this folder in your notebook directory
ONNX_MODEL_PATH = Path(ONNX_OUTPUT_DIR) / "model.onnx"
MODEL_TASK = "sequence-classification"

In [20]:
# Create the output directory if it doesn't exist
Path(ONNX_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Run the export command
# Note: The backslashes allow the command to span multiple lines for readability.
export_command = (
    f"optimum-cli export onnx "
    f"--model {CHECKPOINT_PATH} "
    f"--task {MODEL_TASK} "
    f"{ONNX_OUTPUT_DIR}"
)

# Execute the command and capture output
try:
    result = subprocess.run(export_command, shell=True, check=True, capture_output=True, text=True)
    print("\n✅ Conversion Successful!")
    print(f"ONNX files saved in: {ONNX_OUTPUT_DIR}")
    print("\n--- Command Output ---")
    print(result.stdout)
    if result.stderr:
        print("\n--- Command Errors (if any) ---")
        print(result.stderr)
        
except subprocess.CalledProcessError as e:
    print(f"\n❌ Conversion FAILED! Error:\n{e.stderr}")


✅ Conversion Successful!
ONNX files saved in: OnnxModels/TrainSilOnnx

--- Command Output ---


--- Command Errors (if any) ---
`torch_dtype` is deprecated! Use `dtype` instead!
/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/transformers/models/mobilebert/modeling_mobilebert.py:524: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  torch.tensor(1000),
/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_

In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
from transformers import AutoTokenizer
import onnxruntime as ort # NEW IMPORT for ONNX

# This assumes 'utils.py' is in the same directory
from utils import SegmentFromFile 

# Define the ONNX model file name, typically saved as 'model.onnx'
ONNX_FILE_NAME = "model.onnx"

def evaluate_model_onnx(model_path, validation_data_directory, time_gap, device="cuda"):
    """
    Evaluates an ONNX MobileBERT classifier on validation data using ONNX Runtime.

    Args:
        model_path (str): Path to the directory containing the 'model.onnx' file 
                        and the tokenizer configuration (e.g., mobilebert_onnx_export).
        validation_data_directory (str): Path to the directory of validation CSV files.
        time_gap (float): The time gap used for segmenting the validation data.
        device (str): The device to run inference on ("cuda" or "cpu"). 
                    Note: This selects the ORT Execution Provider.

    Returns:
        dict: A dictionary containing performance metrics (accuracy, f1, precision, recall).
    """
    # --- 1. Load the ONNX Model and Tokenizer ---
    onnx_model_path = os.path.join(model_path, ONNX_FILE_NAME)
    print(f"\n--- Starting Evaluation of ONNX Model: {onnx_model_path} ---")

    if not os.path.isdir(model_path):
        raise FileNotFoundError(f"Model directory not found at: {model_path}")
    if not os.path.exists(onnx_model_path):
        raise FileNotFoundError(f"ONNX model file not found at: {onnx_model_path}")
        
    print("Loading ONNX model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Select Execution Provider based on device argument
    if device == "cuda":
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    else:
        providers = ['CPUExecutionProvider']
        
    # Create the ONNX Runtime session
    session = ort.InferenceSession(onnx_model_path, providers=providers)
    
    # Get input/output names from the ONNX graph
    input_names = [input.name for input in session.get_inputs()]
    output_names = [output.name for output in session.get_outputs()]
    
    print(f"ONNX Inputs: {input_names}")
    print(f"ONNX Outputs: {output_names}")

    # --- 2. Load and Process Validation Data (SAME AS ORIGINAL) ---
    print(f"Loading validation data from: {validation_data_directory}")
    all_chunks = []
    all_labels = []

    csv_files = glob.glob(os.path.join(validation_data_directory, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {validation_data_directory}")

    for file_path in tqdm(csv_files, desc="Processing validation files"):
        filename = os.path.basename(file_path)
        chunks, labels = SegmentFromFile(validation_data_directory, filename, time_gap=time_gap)
        all_chunks.extend(chunks)
        all_labels.extend(labels)

    print(f"Total validation segments loaded: {len(all_chunks)}")

    # --- 3. Format Data to Match Training Input (SAME AS ORIGINAL) ---
    def format_chunk_to_string(chunk):
        tokens = []
        for pair in chunk:
            token1 = f"T{int(pair[0])}"
            token2 = f"G{int(pair[1])}"
            tokens.append(token1)
            tokens.append(token2)
        return " ".join(tokens)

    texts = [format_chunk_to_string(chunk) for chunk in tqdm(all_chunks, desc="Formatting segments")]

    # --- 4. Run Inference in Batches using ONNX Runtime ---
    print("Running inference on validation data...")
    batch_size = 16 
    all_preds = []
    
    num_batches = (len(texts) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(texts), batch_size), total=num_batches, desc="Inference"):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenizer returns NumPy arrays for ONNX (return_tensors="np")
        inputs = tokenizer(
            batch_texts,
            return_tensors="np",
            padding=True,
            truncation=True,
            max_length=512
        )

        # The ONNX model expects int64 type, but the tokenizer may return int32, 
        # so we ensure the correct type for all required inputs.
        # This converts the NumPy array inputs to the required type and format.
        onnx_inputs = {
            name: inputs[name].astype(np.int64) 
            for name in input_names if name in inputs
        }
        
        # Run inference using the ONNX session
        # session.run returns a list of NumPy arrays (outputs)
        onnx_outputs = session.run(output_names, onnx_inputs)
        
        # The first output is the logits (a NumPy array)
        logits = onnx_outputs[0]
        
        # Calculate predictions (argmax across the class dimension)
        preds = np.argmax(logits, axis=-1)

        all_preds.extend(preds.tolist())

    # The probability tracking is removed as it's not straightforward 
    # to extract a specific class probability (e.g., class 1) without
    # knowing the exact ONNX output configuration or applying a custom 
    # softmax layer if needed. We focus on predictions (argmax) for core evaluation.
    # all_probs = [] # REMOVED: Adjusting to ONNX output simplicity

    # --- 5. Calculate and Display Metrics (SAME AS ORIGINAL) ---
    print("\nCalculating metrics...")
    
    # Ensure all_labels is numpy array for consistency
    all_labels_np = np.array(all_labels)
    all_preds_np = np.array(all_preds)
    
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels_np, all_preds_np, average='binary', zero_division=0)
    acc = accuracy_score(all_labels_np, all_preds_np)
    cm = confusion_matrix(all_labels_np, all_preds_np)

    metrics = {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

    print("\n=== Validation Results (ONNX) ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

    print("\n=== Confusion Matrix ===")
    tn, fp, fn, tp = cm.ravel()
    print(f"               Predicted Negative | Predicted Positive")
    print(f"Actual Negative: {tn:<17d}| {fp:<17d}")
    print(f"Actual Positive: {fn:<17d}| {tp:<17d}")
    print("=" * 30)

    return metrics

In [6]:
TIME_GAP_TEST = 83.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low"
# 1. Update the path to the ONNX model directory
onnx_model_directory = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainKiaOnnx" 

# Note: The 'device' will now select the ONNX Execution Provider (e.g., CUDA or CPU)
device_to_use = "cuda" if "cuda" in os.environ.get('DEVICE', 'cuda').lower() and ort.get_device() == 'GPU' else "cpu"

print(f"\n--- Evaluating ONNX checkpoint: {onnx_model_directory} on device: {device_to_use} ---")
# 2. Call the new ONNX evaluation function
metrics = evaluate_model_onnx(
    model_path=onnx_model_directory,  # Path to the directory containing model.onnx
    validation_data_directory=VALIDATION_DATA_DIRECTORY, 
    time_gap=TIME_GAP_TEST,
    device=device_to_use
)


--- Evaluating ONNX checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainKiaOnnx on device: cuda ---

--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainKiaOnnx/model.onnx ---
Loading ONNX model and tokenizer...
ONNX Inputs: ['input_ids', 'attention_mask', 'token_type_ids']
ONNX Outputs: ['logits']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Tesla/Lower Low


Processing validation files: 100%|██████████| 4/4 [00:11<00:00,  2.96s/it]


Total validation segments loaded: 3350


Formatting segments: 100%|██████████| 3350/3350 [00:00<00:00, 9737.35it/s]


Running inference on validation data...


Inference: 100%|██████████| 210/210 [00:12<00:00, 16.76it/s]



Calculating metrics...

=== Validation Results (ONNX) ===
Accuracy:  0.9039
F1 Score:  0.8613
Precision: 0.7788
Recall:    0.9634

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2028             | 284              
Actual Positive: 38               | 1000             


In [29]:
# --- 10. Evaluate a Specific Checkpoint ---
# The path you provided is used here
TIME_GAP_TEST = 105.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Gen/Lower Low"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainGenOnnx"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model_onnx(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainGenOnnx ---

--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainGenOnnx/model.onnx ---
Loading ONNX model and tokenizer...


/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


ONNX Inputs: ['input_ids', 'attention_mask', 'token_type_ids']
ONNX Outputs: ['logits']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Gen/Lower Low


Processing validation files:  25%|██▌       | 1/4 [00:02<00:06,  2.22s/it]/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/utils.py:91: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files:  50%|█████     | 2/4 [00:02<00:02,  1.30s/it]/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/utils.py:91: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files: 100%|██████████| 4/4 [00:07<00:00,  1.89s/it]


Total validation segments loaded: 3287


Formatting segments: 100%|██████████| 3287/3287 [00:00<00:00, 8984.87it/s]


Running inference on validation data...


Inference: 100%|██████████| 206/206 [04:41<00:00,  1.36s/it]


Calculating metrics...

=== Validation Results (ONNX) ===
Accuracy:  0.9909
F1 Score:  0.9851
Precision: 0.9970
Recall:    0.9735

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2264             | 3                
Actual Positive: 27               | 993              


{'accuracy': 0.9908731365987222,
 'f1': 0.9851190476190477,
 'precision': 0.9969879518072289,
 'recall': 0.9735294117647059}

In [27]:
TIME_GAP_TEST = 100.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Kia/Lower Low"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainTeslaOnnx"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model_onnx(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainTeslaOnnx ---

--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainTeslaOnnx/model.onnx ---
Loading ONNX model and tokenizer...
ONNX Inputs: ['input_ids', 'attention_mask', 'token_type_ids']
ONNX Outputs: ['logits']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Kia/Lower Low


/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
Processing validation files: 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


Total validation segments loaded: 3311


Formatting segments: 100%|██████████| 3311/3311 [00:00<00:00, 8878.72it/s]


Running inference on validation data...


Inference: 100%|██████████| 207/207 [04:39<00:00,  1.35s/it]


Calculating metrics...

=== Validation Results (ONNX) ===
Accuracy:  0.9598
F1 Score:  0.9361
Precision: 0.9250
Recall:    0.9475

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2203             | 79               
Actual Positive: 54               | 975              


{'accuracy': 0.959830866807611,
 'f1': 0.9361497839654345,
 'precision': 0.9250474383301708,
 'recall': 0.9475218658892128}

In [30]:
TIME_GAP_TEST = 105.0
VALIDATION_DATA_DIRECTORY = "/home/lisa/Arupreza/UIDS-II/Split_data/Test/Gen/Lower Low"
checkpoint_to_evaluate = "/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainSilOnnx"

print(f"\n--- Evaluating specific checkpoint: {checkpoint_to_evaluate} ---")
evaluate_model_onnx(
    model_path=checkpoint_to_evaluate,
    validation_data_directory=VALIDATION_DATA_DIRECTORY, # From your script's config
    time_gap=TIME_GAP_TEST                             # From your script's config
)


--- Evaluating specific checkpoint: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainSilOnnx ---

--- Starting Evaluation of ONNX Model: /home/lisa/Arupreza/UIDS-II/SFTSrc/bert/OnnxModels/TrainSilOnnx/model.onnx ---
Loading ONNX model and tokenizer...
ONNX Inputs: ['input_ids', 'attention_mask', 'token_type_ids']
ONNX Outputs: ['logits']
Loading validation data from: /home/lisa/Arupreza/UIDS-II/Split_data/Test/Gen/Lower Low


/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
Processing validation files:  25%|██▌       | 1/4 [00:00<00:01,  1.64it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/utils.py:91: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files:  50%|█████     | 2/4 [00:01<00:01,  1.72it/s]/home/lisa/Arupreza/UIDS-II/SFTSrc/bert/utils.py:91: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
Processing validation files: 100%|██████████| 4/4 [00:05<00:00,  1.31s/it]


Total validation segments loaded: 3287


Formatting segments: 100%|██████████| 3287/3287 [00:00<00:00, 9137.48it/s]


Running inference on validation data...


Inference: 100%|██████████| 206/206 [04:41<00:00,  1.37s/it]


Calculating metrics...

=== Validation Results (ONNX) ===
Accuracy:  0.9909
F1 Score:  0.9851
Precision: 0.9970
Recall:    0.9735

=== Confusion Matrix ===
               Predicted Negative | Predicted Positive
Actual Negative: 2264             | 3                
Actual Positive: 27               | 993              


{'accuracy': 0.9908731365987222,
 'f1': 0.9851190476190477,
 'precision': 0.9969879518072289,
 'recall': 0.9735294117647059}